In [ ]:
pip install unsloth

In [ ]:
pip install transformers==4.56.2

In [ ]:
pip install --no-deps trl==0.22.2

In [ ]:
pip install jiwer

In [ ]:
pip install einops

In [ ]:
pip install addict

In [ ]:
pip install easydict

In [ ]:
pip install matplotlib

In [ ]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install jiwer
!pip install einops addict easydict

In [ ]:
# Download DeepSeek OCR model from Hugging Face
from huggingface_hub import snapshot_download

snapshot_download("unsloth/DeepSeek-OCR-2", local_dir="deepseek_ocr2")

In [ ]:
# Load DeepSeek OCR model with FastVisionModel
from unsloth import FastVisionModel
import torch
from transformers import AutoModel
import os

os.environ["UNSLOTH_WARN_UNINITIALIZED"] = '0'

model, tokenizer = FastVisionModel.from_pretrained(
    "./deepseek_ocr2",
    load_in_4bit=True,
    auto_model=AutoModel,
    trust_remote_code=True,
    unsloth_force_compile=True,
    use_gradient_checkpointing="unsloth",
)

In [ ]:
# Load dataset from Hugging Face
from datasets import load_dataset
from huggingface_hub import login
import pandas as pd
from PIL import Image

login()

In [ ]:
# Load dataset from Hugging Face
HF_USERNAME = "avishadilhara"
DATASET_NAME = "sinhala-ocr-lk-acts-1010"

print(f"Loading dataset from Hugging Face...")
dataset = load_dataset(f"{HF_USERNAME}/{DATASET_NAME}")

print(f"Dataset loaded successfully!")
print(f"   Train: {len(dataset['train'])} samples")
print(f"   Eval: {len(dataset['eval'])} samples")
print(f"   Test: {len(dataset['test'])} samples")

# Preview sample
sample = dataset['train'][0]
print(f"\nSample structure:")
print(f"   Image size: {sample['image'].size}")
print(f"   Text length: {len(sample['text'])} chars")
print(f"   Year: {sample['year']}")
print(f"   Preview: {sample['text'][:100]}...")

In [ ]:
# Dataset transformation for training
from datasets import load_dataset
from PIL import Image

instruction = "<image>\nFree OCR."

def transform_sample(examples):
    images = examples['image']
    texts = examples['text']

    is_batch = isinstance(images, list)
    if not is_batch:
        images = [images]
        texts = [texts]

    messages_batch = []

    for img, text in zip(images, texts):
        try:
            if isinstance(img, list):
                img = img[0] if img else None

            if not isinstance(img, Image.Image):
                raise TypeError(f"Expected PIL Image, got {type(img)}")

            max_size = 1024
            if img.width > max_size or img.height > max_size:
                ratio = min(max_size / img.width, max_size / img.height)
                new_size = (int(img.width * ratio), int(img.height * ratio))
                img = img.resize(new_size, Image.LANCZOS)

            msg = [
                {
                    "role": "<|User|>",
                    "content": instruction,
                    "images": [img]
                },
                {
                    "role": "<|Assistant|>",
                    "content": str(text)
                }
            ]
            messages_batch.append(msg)

        except Exception as e:
            print(f"Error processing sample: {e}")
            messages_batch.append([])

    if is_batch:
        return {"messages": messages_batch}
    else:
        return {"messages": messages_batch[0]}


print("Loading dataset...")
dataset = load_dataset("avishadilhara/sinhala-ocr-lk-acts-1010")

print(f"Loaded:")
print(f"   Train: {len(dataset['train'])} samples")
print(f"   Eval: {len(dataset['eval'])} samples")

print("\nApplying transform...")
dataset['train'].set_transform(transform_sample)
dataset['eval'].set_transform(transform_sample)

print(f"\nDatasets ready:")
print(f"   Instruction: '{instruction}'")
print(f"   Roles: '<|User|>' and '<|Assistant|>'")
print(f"   Train: {len(dataset['train'])} samples")
print(f"   Eval: {len(dataset['eval'])} samples")

# Verification
print("\nVerification:")

for split_name in ['train', 'eval']:
    try:
        sample = dataset[split_name][0]

        print(f"\n{split_name.upper()}:")
        print(f"   Sample keys: {list(sample.keys())}")
        print(f"   Messages count: {len(sample['messages'])}")

        if len(sample['messages']) == 2:
            msg0 = sample['messages'][0]
            msg1 = sample['messages'][1]

            print(f"   2 messages found")
            print(f"      Message 0: Role={msg0.get('role')}, Images={len(msg0.get('images', []))}")
            print(f"      Message 1: Role={msg1.get('role')}, Text_length={len(msg1.get('content', ''))} chars")
        else:
            print(f"   ERROR: Expected 2 messages, got {len(sample['messages'])}")

    except Exception as e:
        print(f"   Error accessing {split_name}: {e}")
        import traceback
        traceback.print_exc()

print("\nDataset ready for training!")

In [ ]:
# Configure LoRA for model fine-tuning
model = FastVisionModel.get_peft_model(
    model,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
)

In [ ]:
# Data collator for DeepSeek OCR model
import torch
import math
from dataclasses import dataclass
from typing import Dict, List, Any, Tuple
from PIL import Image, ImageOps
from torch.nn.utils.rnn import pad_sequence
import io

from deepseek_ocr2.modeling_deepseekocr2 import (
    format_messages,
    text_encode,
    BasicImageTransform,
    dynamic_preprocess,
)

@dataclass
class DeepSeekOCR2DataCollator:
    """Data collator for OCR training."""

    tokenizer: Any
    model: Any
    image_size: int = 768
    base_size: int = 1024
    crop_mode: bool = True
    image_token_id: int = 128815
    train_on_responses_only: bool = True

    def __init__(
        self,
        tokenizer,
        model,
        image_size: int = 768,
        base_size: int = 1024,
        crop_mode: bool = True,
        train_on_responses_only: bool = True,
    ):
        self.tokenizer = tokenizer
        self.model = model
        self.image_size = image_size
        self.base_size = base_size
        self.crop_mode = crop_mode
        self.image_token_id = 128815
        self.dtype = model.dtype
        self.train_on_responses_only = train_on_responses_only

        self.image_transform = BasicImageTransform(
            mean=(0.5, 0.5, 0.5),
            std=(0.5, 0.5, 0.5),
            normalize=True
        )
        self.patch_size = 16
        self.downsample_ratio = 4

        if hasattr(tokenizer, 'bos_token_id') and tokenizer.bos_token_id is not None:
            self.bos_id = tokenizer.bos_token_id
        else:
            self.bos_id = 0
            print(f"Warning: tokenizer has no bos_token_id, using default: {self.bos_id}")

    def deserialize_image(self, image_data) -> Image.Image:
        """Convert image data to PIL Image in RGB mode."""
        if isinstance(image_data, Image.Image):
            return image_data.convert("RGB")
        elif isinstance(image_data, dict) and 'bytes' in image_data:
            image_bytes = image_data['bytes']
            image = Image.open(io.BytesIO(image_bytes))
            return image.convert("RGB")
        else:
            raise ValueError(f"Unsupported image format: {type(image_data)}")

    def calculate_image_token_count(self, image: Image.Image, crop_ratio: Tuple[int, int]) -> int:
        """Calculate the number of tokens this image will generate."""
        num_queries = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
        num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)

        width_crop_num, height_crop_num = crop_ratio

        if self.crop_mode:
            img_tokens = num_queries_base * num_queries_base + 1
            if width_crop_num > 1 or height_crop_num > 1:
                img_tokens += (num_queries * width_crop_num) * (num_queries * height_crop_num)
        else:
            img_tokens = num_queries * num_queries + 1

        return img_tokens

    def process_image(self, image: Image.Image) -> Tuple[List, List, List, List, Tuple[int, int]]:
        """Process a single image based on crop_mode and size thresholds."""
        images_list = []
        images_crop_list = []
        images_spatial_crop = []

        if self.crop_mode:
            if image.size[0] <= 768 and image.size[1] <= 768:
                crop_ratio = (1, 1)
                images_crop_raw = []
            else:
                images_crop_raw, crop_ratio = dynamic_preprocess(
                    image, min_num=2, max_num=6,
                    image_size=self.image_size, use_thumbnail=False
                )

            global_view = ImageOps.pad(
                image, (self.base_size, self.base_size),
                color=tuple(int(x * 255) for x in self.image_transform.mean)
            )
            images_list.append(self.image_transform(global_view).to(self.dtype))

            width_crop_num, height_crop_num = crop_ratio
            images_spatial_crop.append([width_crop_num, height_crop_num])

            if width_crop_num > 1 or height_crop_num > 1:
                for crop_img in images_crop_raw:
                    images_crop_list.append(
                        self.image_transform(crop_img).to(self.dtype)
                    )

            num_queries = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
            num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)

            tokenized_image = ([self.image_token_id] * num_queries_base) * num_queries_base
            tokenized_image += [self.image_token_id]

            if width_crop_num > 1 or height_crop_num > 1:
                tokenized_image += ([self.image_token_id] * (num_queries * width_crop_num)) * (
                    num_queries * height_crop_num)

        else:
            crop_ratio = (1, 1)
            images_spatial_crop.append([1, 1])

            if self.base_size <= 768:
                resized_image = image.resize((self.base_size, self.base_size), Image.LANCZOS)
                images_list.append(self.image_transform(resized_image).to(self.dtype))
            else:
                global_view = ImageOps.pad(
                    image, (self.base_size, self.base_size),
                    color=tuple(int(x * 255) for x in self.image_transform.mean)
                )
                images_list.append(self.image_transform(global_view).to(self.dtype))

            num_queries = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)
            tokenized_image = ([self.image_token_id] * num_queries) * num_queries
            tokenized_image += [self.image_token_id]

        return images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio

    def process_single_sample(self, messages: List[Dict]) -> Dict[str, Any]:
        """Process a single conversation into model inputs."""

        images = []
        for message in messages:
            if "images" in message and message["images"]:
                for img_data in message["images"]:
                    if img_data is not None:
                        pil_image = self.deserialize_image(img_data)
                        images.append(pil_image)

        if not images:
            raise ValueError("No images found in sample.")

        tokenized_str = []
        images_seq_mask = []
        images_list, images_crop_list, images_spatial_crop = [], [], []

        prompt_token_count = -1
        assistant_started = False
        image_idx = 0

        tokenized_str.append(self.bos_id)
        images_seq_mask.append(False)

        for message in messages:
            role = message["role"]
            content = message["content"]

            if role == "<|Assistant|>":
                if not assistant_started:
                    prompt_token_count = len(tokenized_str)
                    assistant_started = True

                content = f"{content.strip()} {self.tokenizer.eos_token}"

            text_splits = content.split('<image>')

            for i, text_sep in enumerate(text_splits):
                tokenized_sep = text_encode(self.tokenizer, text_sep, bos=False, eos=False)
                tokenized_str.extend(tokenized_sep)
                images_seq_mask.extend([False] * len(tokenized_sep))

                if i < len(text_splits) - 1:
                    if image_idx >= len(images):
                        raise ValueError(
                            f"Mismatch: Found '<image>' token but no corresponding image."
                        )

                    image = images[image_idx]
                    img_list, crop_list, spatial_crop, tok_img, _ = self.process_image(image)

                    images_list.extend(img_list)
                    images_crop_list.extend(crop_list)
                    images_spatial_crop.extend(spatial_crop)

                    tokenized_str.extend(tok_img)
                    images_seq_mask.extend([True] * len(tok_img))

                    image_idx += 1

        if image_idx != len(images):
            raise ValueError(
                f"Mismatch: Found {len(images)} images but only {image_idx} '<image>' tokens."
            )

        if not assistant_started:
            print("Warning: No assistant message found. Masking all tokens.")
            prompt_token_count = len(tokenized_str)

        images_ori = torch.stack(images_list, dim=0)
        images_spatial_crop_tensor = torch.tensor(images_spatial_crop, dtype=torch.long)

        if images_crop_list:
            images_crop = torch.stack(images_crop_list, dim=0)
        else:
            images_crop = torch.zeros((1, 3, self.base_size, self.base_size), dtype=self.dtype)

        return {
            "input_ids": torch.tensor(tokenized_str, dtype=torch.long),
            "images_seq_mask": torch.tensor(images_seq_mask, dtype=torch.bool),
            "images_ori": images_ori,
            "images_crop": images_crop,
            "images_spatial_crop": images_spatial_crop_tensor,
            "prompt_token_count": prompt_token_count,
        }

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        """Collate batch of samples."""
        batch_data = []

        for feature in features:
            try:
                processed = self.process_single_sample(feature['messages'])
                batch_data.append(processed)
            except Exception as e:
                print(f"Error processing sample: {e}")
                continue

        if not batch_data:
            raise ValueError("No valid samples in batch")

        input_ids_list = [item['input_ids'] for item in batch_data]
        images_seq_mask_list = [item['images_seq_mask'] for item in batch_data]
        prompt_token_counts = [item['prompt_token_count'] for item in batch_data]

        input_ids = pad_sequence(input_ids_list, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        images_seq_mask = pad_sequence(images_seq_mask_list, batch_first=True, padding_value=False)

        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        labels[images_seq_mask] = -100

        if self.train_on_responses_only:
            for idx, prompt_count in enumerate(prompt_token_counts):
                if prompt_count > 0:
                    labels[idx, :prompt_count] = -100

        attention_mask = (input_ids != self.tokenizer.pad_token_id).long()

        images_batch = []
        for item in batch_data:
            images_batch.append((item['images_crop'], item['images_ori']))

        images_spatial_crop = torch.cat([item['images_spatial_crop'] for item in batch_data], dim=0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "images": images_batch,
            "images_seq_mask": images_seq_mask,
            "images_spatial_crop": images_spatial_crop,
        }

In [ ]:
# Clear GPU cache
torch.cuda.empty_cache()

In [ ]:
# Training configuration and execution
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, TrainerCallback
from unsloth import is_bf16_supported

FastVisionModel.for_training(model)

# Data collator
data_collator = DeepSeekOCR2DataCollator(
    tokenizer=tokenizer,
    model=model,
    image_size=768,
    base_size=1024,
    crop_mode=True,
    train_on_responses_only=True,
)

# Training arguments
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    num_train_epochs=20,

    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_steps=10,

    optim="adamw_8bit",
    weight_decay=0.001,

    fp16=False,
    bf16=True,

    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    logging_steps=5,
    report_to="none",

    seed=3407,
    dataloader_num_workers=2,
    remove_unused_columns=False,

    output_dir="./sinhala_ocr_outputs",
)

# Create trainer
trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    data_collator=data_collator,
    train_dataset=dataset['train'],
    eval_dataset=dataset['eval'],
    args=training_args,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=1),
    ]
)

print("Trainer configured!")
print(f"Training samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['eval'])}")
print(f"Test samples: {len(dataset['test'])}")

In [ ]:
# Start training
print("Starting training...")
trainer_stats = trainer.train()

In [ ]:
# Resume training from checkpoint
trainer_stats = trainer.train(resume_from_checkpoint="sinhala_ocr_outputs/checkpoint-25")
print("Training completed!")

In [ ]:
# Save trained model and tokenizer
model.save_pretrained("sinhala_deepseek_ocr_lora")
tokenizer.save_pretrained("sinhala_deepseek_ocr_lora")

print("Model saved locally!")

In [ ]:
# Create zip file of trained model
import os
import shutil
from datetime import datetime

model_dir = "sinhala_deepseek_ocr_lora"

if os.path.exists(model_dir):
    model_size_mb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, fns in os.walk(model_dir) for f in fns
    ) / (1024 * 1024)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_name = f"sinhala_ocr_lora_{timestamp}"

    print(f"Zipping trained model ({model_size_mb:.1f} MB)...")
    shutil.make_archive(zip_name, 'zip', '.', model_dir)

    zip_size_mb = os.path.getsize(f"{zip_name}.zip") / (1024 * 1024)
    compression_ratio = (1 - zip_size_mb / model_size_mb) * 100

    print(f"Created: {zip_name}.zip ({zip_size_mb:.1f} MB)")
    print(f"Compression: {compression_ratio:.1f}% reduction")
else:
    print(f"Model directory '{model_dir}' not found. Save the model first!")

In [ ]:
# Test model inference on a single sample
from jiwer import cer
import os
import glob

FastVisionModel.for_training(model, False)

os.makedirs("./test_output", exist_ok=True)

test_sample = dataset['test'][4]
test_sample['image'].save("test_img.jpg")

print("Testing OCR Model")
print("-" * 60)
print("Running inference on test sample...")

model.infer(
    tokenizer,
    prompt="<image> OCR.",
    image_file="test_img.jpg",
    output_path="./test_output",
    base_size=1024,
    image_size=768,
    crop_mode=True,
    save_results=True,
)

# Read prediction from output file
mmd_files = sorted(glob.glob("./test_output/*.mmd"))
if mmd_files:
    with open(mmd_files[-1], "r", encoding="utf-8") as f:
        prediction = f.read().strip()
else:
    txt_files = sorted(glob.glob("./test_output/*.txt"))
    if txt_files:
        with open(txt_files[-1], "r", encoding="utf-8") as f:
            prediction = f.read().strip()
    else:
        print("No output file found!")
        prediction = ""

if prediction:
    print("\nInference Complete!")
    print("-" * 60)
    print("GROUND TRUTH (first 300 chars):")
    print(test_sample['text'][:300])
    print("\n" + "-" * 60)
    print("PREDICTED (first 300 chars):")
    print(prediction[:300])

    error_rate = cer(test_sample['text'], prediction)
    accuracy = (1 - error_rate) * 100

    print("\n" + "-" * 60)
    print("METRICS")
    print("-" * 60)
    print(f"Character Error Rate: {error_rate:.2%}")
    print(f"Character Accuracy: {accuracy:.2f}%")
    print(f"Ground Truth Length: {len(test_sample['text'])} chars")
    print(f"Predicted Length: {len(prediction)} chars")
else:
    print("No prediction generated.")

In [ ]:
# Zip trained model and best checkpoint for download
import os
import shutil
from datetime import datetime

def zip_trained_model(
    model_dir="sinhala_deepseek_ocr_lora",
    output_dir="./",
    include_timestamp=True,
    also_zip_best_checkpoint=True,
    checkpoint_dir="./sinhala_ocr_outputs"
):
    """Zip the final trained model and optionally the best checkpoint."""

    print("=" * 70)
    print("ZIPPING TRAINED MODEL")
    print("=" * 70)

    zipped_files = []

    # Zip trained model (LoRA adapters)
    if os.path.exists(model_dir):
        print(f"\nZipping trained model: {model_dir}/")

        model_size_mb = sum(
            os.path.getsize(os.path.join(dirpath, filename))
            for dirpath, dirnames, filenames in os.walk(model_dir)
            for filename in filenames
        ) / (1024 * 1024)

        print(f"Original size: {model_size_mb:.1f} MB")

        if include_timestamp:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            zip_name = f"sinhala_ocr_lora_{timestamp}"
        else:
            zip_name = "sinhala_ocr_lora_final"

        zip_path = os.path.join(output_dir, zip_name)

        print(f"Zipping to: {zip_name}.zip")
        shutil.make_archive(
            base_name=zip_path,
            format='zip',
            root_dir=os.path.dirname(model_dir) if os.path.dirname(model_dir) else '.',
            base_dir=os.path.basename(model_dir)
        )

        final_zip = f"{zip_path}.zip"
        zip_size_mb = os.path.getsize(final_zip) / (1024 * 1024)
        compression_ratio = (1 - zip_size_mb / model_size_mb) * 100

        print(f"Created: {os.path.basename(final_zip)}")
        print(f"Zip size: {zip_size_mb:.1f} MB")
        print(f"Compression: {compression_ratio:.1f}% reduction")

        zipped_files.append({
            'name': os.path.basename(final_zip),
            'path': final_zip,
            'size_mb': zip_size_mb,
            'type': 'LoRA Model'
        })
    else:
        print(f"\nModel directory not found: {model_dir}")

    # Zip best checkpoint (optional)
    if also_zip_best_checkpoint:
        print(f"\nLooking for best checkpoint in: {checkpoint_dir}/")

        checkpoints = []
        if os.path.exists(checkpoint_dir):
            for item in os.listdir(checkpoint_dir):
                item_path = os.path.join(checkpoint_dir, item)
                if os.path.isdir(item_path) and item.startswith('checkpoint-'):
                    checkpoints.append(item_path)

        if checkpoints:
            checkpoints.sort(key=os.path.getmtime, reverse=True)
            best_checkpoint = checkpoints[0]

            print(f"Found best checkpoint: {os.path.basename(best_checkpoint)}/")

            checkpoint_size_mb = sum(
                os.path.getsize(os.path.join(dirpath, filename))
                for dirpath, dirnames, filenames in os.walk(best_checkpoint)
                for filename in filenames
            ) / (1024 * 1024)

            print(f"Original size: {checkpoint_size_mb:.1f} MB")

            if include_timestamp:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                checkpoint_zip_name = f"best_checkpoint_{timestamp}"
            else:
                checkpoint_zip_name = f"best_{os.path.basename(best_checkpoint)}"

            checkpoint_zip_path = os.path.join(output_dir, checkpoint_zip_name)

            print(f"Zipping to: {checkpoint_zip_name}.zip")
            shutil.make_archive(
                base_name=checkpoint_zip_path,
                format='zip',
                root_dir=checkpoint_dir,
                base_dir=os.path.basename(best_checkpoint)
            )

            final_checkpoint_zip = f"{checkpoint_zip_path}.zip"
            checkpoint_zip_size_mb = os.path.getsize(final_checkpoint_zip) / (1024 * 1024)
            checkpoint_compression_ratio = (1 - checkpoint_zip_size_mb / checkpoint_size_mb) * 100

            print(f"Created: {os.path.basename(final_checkpoint_zip)}")
            print(f"Zip size: {checkpoint_zip_size_mb:.1f} MB")
            print(f"Compression: {checkpoint_compression_ratio:.1f}% reduction")

            zipped_files.append({
                'name': os.path.basename(final_checkpoint_zip),
                'path': final_checkpoint_zip,
                'size_mb': checkpoint_zip_size_mb,
                'type': 'Best Checkpoint'
            })
        else:
            print(f"No checkpoints found in {checkpoint_dir}")

    # Summary
    print("\n" + "=" * 70)
    print("ZIPPING COMPLETE")
    print("=" * 70)

    if zipped_files:
        print(f"\nCreated {len(zipped_files)} zip file(s):")
        total_size = 0

        for i, zf in enumerate(zipped_files, 1):
            print(f"\n{i}. {zf['name']}")
            print(f"   Type: {zf['type']}")
            print(f"   Size: {zf['size_mb']:.1f} MB")
            print(f"   Path: {zf['path']}")
            total_size += zf['size_mb']

        print(f"\nTotal size: {total_size:.1f} MB")
        print("=" * 70)

    return zipped_files


# Run zipping
zipped = zip_trained_model(
    model_dir="sinhala_deepseek_ocr_lora",
    output_dir="./",
    include_timestamp=True,
    also_zip_best_checkpoint=True,
    checkpoint_dir="./sinhala_ocr_outputs"
)